# Before AI Can See: How We Manipulate Images
### Computer Vision Series · Part 02 — Companion OpenCV Notebook

**From image arrays to a detected tennis ball, without training a neural network.**

This notebook follows the Medium article's story: first learn how transformations change the pixel grid, then build a classical computer-vision pipeline. All examples run on a **synthetic demonstration image**, so no external dataset is needed. You can optionally use your own image near the end.

**Dependencies:** `numpy`, `opencv-python`, `matplotlib`. If needed, run `%pip install numpy opencv-python matplotlib` in a notebook cell.

## 1 · Imports and a helper for displaying OpenCV images

OpenCV stores colour images as **BGR** arrays. Matplotlib expects **RGB**. We convert only for display; our processing continues in BGR unless stated otherwise.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV:", cv2.__version__)

def show_images(images, titles, figsize=None, cmap=None):
    count = len(images)
    fig, axes = plt.subplots(1, count, figsize=figsize or (4.5 * count, 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 3:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(image, cmap=cmap or "gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2 · Create a reproducible example image

The scene contains a green-yellow tennis ball, a coffee mug, and background objects. This is a **controlled synthetic scene**, not a demonstration of robust detection in the real world. Its purpose is to make the processing stages easy to inspect.

In [ ]:
height, width = 600, 800
image = np.full((height, width, 3), (210, 217, 224), dtype=np.uint8)  # BGR
# Table and subtle background texture
cv2.rectangle(image, (0, 400), (799, 599), (160, 190, 208), -1)
cv2.line(image, (0, 400), (799, 400), (125, 150, 165), 4)
cv2.rectangle(image, (80, 310), (240, 425), (90, 115, 155), -1)  # books
cv2.rectangle(image, (105, 285), (255, 310), (190, 150, 100), -1)
# Coffee mug, deliberately neutral-coloured
cv2.rectangle(image, (585, 275), (700, 425), (235, 235, 235), -1)
cv2.rectangle(image, (585, 275), (700, 425), (130, 130, 130), 4)
cv2.ellipse(image, (704, 343), (32, 46), 0, -85, 85, (130, 130, 130), 9)
# Tennis ball: hue around 37 in OpenCV's 0–179 hue scale
ball_center = (400, 350)
ball_radius = 85
cv2.circle(image, ball_center, ball_radius, (25, 220, 180), -1)  # BGR
cv2.circle(image, ball_center, ball_radius, (20, 155, 130), 3)
cv2.ellipse(image, (385, 350), (35, 80), 20, -100, 100, (230, 240, 235), 5)
cv2.ellipse(image, (415, 350), (35, 80), 20, 80, 280, (230, 240, 235), 5)
print("Image shape (height, width, channels):", image.shape)
show_images([image], ["Our demonstration scene"])

## 3 · Resizing and interpolation

Resizing creates a new pixel grid. The interpolation method controls how values are estimated. For reduction, `INTER_AREA` is often a good choice; `INTER_LINEAR` is a common general-purpose option. Enlarging with `INTER_NEAREST` makes the pixel blocks obvious.

In [ ]:
smaller = cv2.resize(image, (400, 300), interpolation=cv2.INTER_AREA)
blocky = cv2.resize(smaller, (800, 600), interpolation=cv2.INTER_NEAREST)
smooth = cv2.resize(smaller, (800, 600), interpolation=cv2.INTER_LINEAR)
show_images([image, blocky, smooth], ["Original", "Shrink + nearest-neighbour enlarge", "Shrink + linear enlarge"], figsize=(15, 4))

## 4 · Aspect ratio: stretching versus letterboxing

Our original is **800 × 600** (4:3). Forcing it into **224 × 224** changes proportions. Letterboxing scales the image to fit inside the square and pads the unused space. In OpenCV, resize arguments are `(width, height)`; NumPy image shapes are `(height, width, channels)`.

In [ ]:
def letterbox(bgr, target_size=(224, 224), pad_color=(110, 110, 110)):
    target_w, target_h = target_size
    h, w = bgr.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = round(w * scale), round(h * scale)
    resized = cv2.resize(bgr, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR)
    left = (target_w - new_w) // 2
    right = target_w - new_w - left
    top = (target_h - new_h) // 2
    bottom = target_h - new_h - top
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_color)
    return padded, scale, (left, top)

stretched = cv2.resize(image, (224, 224))
padded, scale, offset = letterbox(image)
print("Stretched shape:", stretched.shape)
print("Letterboxed shape:", padded.shape, "| scale:", round(scale, 3), "| padding origin:", offset)
show_images([stretched, padded], ["Forced to 224 × 224 (distorted)", "224 × 224 with proportions preserved"], figsize=(9, 4))

**Important:** Letterboxing preserves geometry, but padding and scale still change the input distribution. Use the preprocessing procedure expected by your actual model. If you later draw detections on the original image, undo the scale and padding to recover original coordinates.

## 5 · Cropping, flipping and rotation

Cropping selects a **region of interest** (ROI). Flipping changes orientation; rotation moves pixel positions and can introduce empty borders. Such transformations can be useful for data augmentation, but only when they preserve the meaning of the training label (for example, mirrored text may no longer have the same meaning).

In [ ]:
crop = image[230:480, 270:530]  # rows y, then columns x
flipped = cv2.flip(image, 1)  # 1 = horizontal, 0 = vertical
h, w = image.shape[:2]
rotation_matrix = cv2.getRotationMatrix2D((w / 2, h / 2), 25, 1.0)  # positive = counterclockwise
rotated = cv2.warpAffine(image, rotation_matrix, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(100, 100, 100))
show_images([image, crop, flipped, rotated], ["Original", "Cropped ROI", "Horizontal flip", "25° counterclockwise"], figsize=(17, 4))
print("Crop shape:", crop.shape)

## 6 · HSV colour thresholding

We want to identify the green-yellow tennis ball using a range of HSV values. In OpenCV, **H is typically 0–179** and S/V are **0–255** for 8-bit images. Thresholds below are tuned for our synthetic example and **will need adjustment** for real photographs.

In [ ]:
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
lower = np.array([30, 80, 80], dtype=np.uint8)
upper = np.array([80, 255, 255], dtype=np.uint8)
mask = cv2.inRange(hsv, lower, upper)
segmented = cv2.bitwise_and(image, image, mask=mask)
print("Ball-centre HSV:", hsv[ball_center[1], ball_center[0]])
show_images([image, mask, segmented], ["Original BGR image", "Binary colour mask", "Masked pixels"], figsize=(13, 4))

## 7 · Clean the mask using morphology

A colour threshold can produce isolated white specks and small holes. **Opening** (erosion followed by dilation) removes small bright noise; **closing** (dilation followed by erosion) fills small dark gaps. A 5×5 elliptical structuring element works for this example. This doesn't guarantee noise-free masks in arbitrary scenes.

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)
show_images([mask, cleaned], ["Before cleaning", "After opening + closing"], figsize=(9, 4))

## 8 · Find contours and draw a bounding box

`findContours` extracts boundaries of connected foreground regions in the binary mask. We filter out small contours by area, then draw the bounding box around the **largest remaining region**. That is a heuristic, not semantic understanding: another similarly coloured object could be selected instead.

In [ ]:
contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
candidates = [c for c in contours if cv2.contourArea(c) > 500]
result = image.copy()
if candidates:
    ball_contour = max(candidates, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(ball_contour)
    cv2.rectangle(result, (x, y), (x + bw, y + bh), (0, 160, 0), 3)
    cv2.putText(result, "Tennis ball candidate", (max(x - 20, 0), max(y - 12, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 120, 0), 2)
    print(f"Bounding box (x, y, width, height): {(x, y, bw, bh)}")
    print(f"Contour area: {cv2.contourArea(ball_contour):.1f} pixels²")
else:
    print("No contour above the minimum area threshold was found.")
show_images([image, cleaned, result], ["Original", "Cleaned binary mask", "Contour-based localisation"], figsize=(14, 4))

## 9 · Save the result

The output image can be used in a GitHub README or Medium post. `cv2.imwrite` writes the BGR image correctly; don't convert to RGB before saving with OpenCV.

In [ ]:
output_path = "tennis_ball_detected.png"
success = cv2.imwrite(output_path, result)
print("Saved:", output_path, "| success:", success)

## 10 · Try your own photograph (optional)

Change the path to a real local image. Because the mask was calibrated on a synthetic scene, you may need to adjust the HSV limits, morphology kernel and area threshold. Use a photo with a bright tennis ball for the best chance of success.

In [ ]:
# Uncomment and change the filename to experiment with your own image.
# photo = cv2.imread("my_tennis_ball.jpg")
# if photo is None:
#     raise FileNotFoundError("Check the image path.")
# photo_hsv = cv2.cvtColor(photo, cv2.COLOR_BGR2HSV)
# photo_mask = cv2.inRange(photo_hsv, lower, upper)
# show_images([photo, photo_mask], ["Your photo", "Your mask"])

## What this pipeline can and cannot do

**What it demonstrates:** image arrays, resizing, interpolation, letterboxing, cropping, flips, rotation, HSV thresholding, morphology, contours and bounding boxes.

**What it does not do:** recognise tennis balls in arbitrary scenes. A similarly coloured object may produce a false positive; changes in lighting may cause missed detections; contours do not know object identity. For more complex and variable settings, learned object detectors may be more suitable.

### Experiment ideas
1. Change the resize interpolation and zoom in to inspect differences.
2. Change the target size to `(320, 192)` and inspect letterboxing.
3. Rotate by 45° and inspect border artifacts.
4. Tighten or widen the HSV threshold and observe what disappears or appears.
5. Add a second similarly coloured circle to the synthetic image and see which contour gets selected.

**Next in the series:** *Teaching Machines to See: Neural Networks Behind Computer Vision.*